<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Operaciones de Aprendizaje Automático III**
# **Clase 3: Ejercicio Instruction Tuning**

Usamos Qwen2.5-0.5B, el modelo base, sin la variante -Instruct, es un modelo que
solo sabe continuar texto. Si le escribimos "Suma 17 y 4. Responde solo el resultado.", lo más
probable es que siga escribiendo más ejercicios de matemática en vez de contestar.

Lo que queremos hacer es:

Aplicar Instruction Tuning para enseñarle a obedecer el formato de la respuesta. Para demostrarlo separamos la evaluación en tres niveles sobre la misma salida:

* **contiene**: la respuesta aparece en algún lugar de la salida (el modelo parece saber)
* **estricto**: aparece sin contar lo que es copia del enunciado (el modelo sabe)
* **exacto**: la salida es exactamente la respuesta (el modelo obedece)

La distancia entre **estricto** y **exacto** es lo que aporta el instruction tuning

Instalamos lo necesario

In [ ]:
!pip -q install -U "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0" "mlflow==2.21.3"
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


Importamos lo necesario

In [ ]:
import torch, json, re, os, gc, time, math, random, hashlib
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
import mlflow

Definimos el modelo base, no usamos el que tiene instruct

In [ ]:
BASE = "Qwen/Qwen2.5-0.5B"

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("clase3-instruction-tuning")

<Experiment: artifact_location='/content/mlruns/1', creation_time=1789500482347, experiment_id='1', last_update_time=1789500482347, lifecycle_stage='active', name='clase3-instruction-tuning', tags={}>

In [ ]:
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay gpu")

gpu: Tesla T4


### **a) Plantilla**

* un modelo base no tiene plantilla de conversación.
* no hay `<|im_start|>` ni roles, porque nunca
fue entrenado con eso
* la plantilla la proponemos nosotros y es parte del artefacto:
quien cargue este adaptador y use otra plantilla obtiene algo que no es util

```
### Instrucción:
{instrucción}

### Respuesta:
{respuesta}
```

Se corta con el token de fin, esto es justamente lo que el modelo tiene que aprender a emitir, sin
eso sigue escribiendo para siempre

In [ ]:
PLANTILLA = "### Instrucción:\n{instr}\n\n### Respuesta:\n"

El prompt va enmascarado con -100, la pérdida se calcula solo sobre la respuesta, si no la mayor parte del gradiente entrena al modelo a escribir instrucciones.

El modelo recibe instr + respuesta, pero la función hace que la pérdida (loss) se calcule solamente sobre la respuesta, porque queremos enseñar al modelo:dada esta instrucción, genera esta respuesta, no queremos que el modelo aprenda a reconstruir la instrucción

In [ ]:
def armar(instr, resp):
    p = tok(PLANTILLA.format(instr=instr), add_special_tokens=False)["input_ids"]
    r = tok(resp, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    return {"input_ids": p + r, "labels": [-100]*len(p) + r}

In [ ]:
p = [21, 4, 3, 29]
r = [2, 3, 9]
[21, 4, 3, 29, 2, 3 ,9]
[-100, -100, -100, -100, 2, 3, 9]

[21, 4, 3, 29, 2, 3, 9]

El modelo base no sabe parar, lo recortamos desde el primer recorte razonable

In [ ]:
def limpiar(salida):
    s = salida.split("###")[0].split("\n\n")[0].strip()
    return s

In [ ]:
def _n(s):
    return re.sub(r"\s+", " ", s.strip()).strip().rstrip(".")

El JSON se compara sin espacios, la clave es la estructura no si el modelo puso un espacio después de los dos puntos: (contiene, exacto)

In [ ]:
def niveles(salida, esperada):
    s, e = _n(limpiar(salida)), _n(esperada)
    if e.startswith("{"):
        s, e = s.replace(" ", ""), e.replace(" ", "")
    return (int(e in s or e in _n(salida).replace(" ", "")), int(s == e))

La puntuación se normaliza en los dos lados: en el enunciado la fecha viene pegada al signo de cierre ("2015-03-05?") y si no se separa

In [ ]:
def _sin_punt(s):
    return _n(re.sub(r"""[¿?¡!.,;:()\"']""", " ", s))

Quitamos de la salida todo fragmento que sea copia textual del enunciado, empezando por los más largos

In [ ]:
def sin_eco(salida, instr):
    s = _sin_punt(limpiar(salida))
    palabras = _sin_punt(instr).split()
    for k in range(len(palabras), 0, -1):
        for i in range(len(palabras) - k + 1):
            frag = " ".join(palabras[i:i+k])
            if len(frag) >= 4 and frag in s:
                s = s.replace(frag, " ")
    return _n(s)

In [ ]:
def contiene_estricto(salida, esperada, instr):
    return int(_sin_punt(esperada) in sin_eco(salida, instr))

In [ ]:
assert niveles("21", "21") == (1, 1)
assert niveles("El resultado es 21.", "21") == (1, 0) # sabe pero no obedece
assert niveles("no sé", "21") == (0, 0)
assert niveles("21\n\n### Instrucción: otra cosa", "21") == (1, 1)   # se recorta lo que no sirve
assert niveles('{"nombre": "Ana", "edad": 31}', '{"nombre":"Ana","edad":31}') == (1, 1)
assert niveles("HOLA MUNDO", "HOLA MUNDO") == (1, 1)
assert niveles("hola mundo", "HOLA MUNDO") == (0, 0)   # mayusculas, si importan aqui

El nivel estricto: repetir el enunciado no cuenta como saber la respuesta

In [ ]:
i1 = "¿De qué año es 2015-03-05? Solo el número."
assert niveles("2015-03-05", "2015")[0] == 1    # contiene lo daba por bueno
assert contiene_estricto("2015-03-05", "2015", i1) == 0    # y era solo un eco
assert contiene_estricto("2015", "2015", i1) == 1

In [ ]:
i3 = "Responde solo la primera palabra de: el tren llegó tarde"
assert contiene_estricto("el tren llegó tarde", "el", i3) == 0 # repetir la frase entera
assert contiene_estricto("el", "el", i3) == 1

* Los tres niveles se acaban de definir y ya funcionan sobre los casos de prueba
* Vamos a definir un catálogo de casos cuyo veredicto se fija a mano, midiendo en cuántos cada nivel coincide con ese veredicto.
+ La pregunta que el catálogo decide es: el modelo resolvió la tarea?
* Se reporta **acuerdo** y **kappa de Cohen**, el acuerdo solo no alcanza: si el 80 % del catálogo es de una clase, una métrica que siempre dice esa clase acierta el 80 % sin medir nada, aqui entra Kappa que descuenta
justamente ese acuerdo por casualidad

### **b) Catálogo de validación**


Cada entrada es: (etiqueta del modo, instrucción, salida del modelo, respuesta esperada, veredicto) esta lista es un artefacto

In [ ]:
CATALOGO = [
 ("limpio", "Suma 36 y 15. Responde solo el resultado.", "51", "51", True),
 ("verboso", "Suma 36 y 15. Responde solo el resultado.", "El resultado es 51.", "51", True),
 ("con cuentas", "Suma 36 y 15. Responde solo el resultado.", "36 + 15 = 51", "51", True),
 ("sin resolver", "Suma 36 y 15. Responde solo el resultado.", "36 + 15", "51", False),
 ("eco de fecha", "¿De qué año es 2015-03-05? Solo el número.", "2015-03-05", "2015", False),
 ("limpio", "¿De qué año es 2015-03-05? Solo el número.", "2015", "2015", True),
 ("eco de frase", "Responde solo la primera palabra de: el tren llegó tarde",
                   "el tren llegó tarde", "el", False),
 ("limpio", "Responde solo la primera palabra de: el tren llegó tarde", "el", "el", True),
 ("verboso", "Responde solo la primera palabra de: el tren llegó tarde",
                   "La primera palabra es el.", "el", True),
 ("limpio", "¿Cuántas palabras tiene esta frase? Responde solo el número: el informe está listo",
                   "4", "4", True),
 ("verboso", "¿Cuántas palabras tiene esta frase? Responde solo el número: el informe está listo",
                   "La frase tiene 4 palabras.", "4", True),
 ("eco de frase", "¿Cuántas palabras tiene esta frase? Responde solo el número: el informe está listo",
                   "el informe está listo", "4", False),
 ("limpio", "Pon este texto en mayúsculas: el informe está listo",
                   "EL INFORME ESTÁ LISTO", "EL INFORME ESTÁ LISTO", True),
 ("sin obedecer",  "Pon este texto en mayúsculas: el informe está listo",
                   "el informe está listo", "EL INFORME ESTÁ LISTO", False),
 ("limpio", "¿La palabra montaña contiene la letra a? Responde solo SI o NO.", "SI", "SI", True),
 # La respuesta correcta aparece, pero el modelo afirma otra cosa (falso positivo)
 ("afirma otra",   "¿Cuántas letras tiene la palabra molino? Responde solo el número.",
                   "Tiene 7 letras, no 6 como muchos creen.", "6", False),
]

**Kappa de Cohen** para dos etiquetados binarios sobre los mismos casos

El coeficiente Kappa de Cohen mide cuánto coinciden dos evaluadores o modelos, descontando la coincidencia que podría ocurrir simplemente por azar.

In [ ]:
def kappa_cohen(a, b):
    n = len(a)
    po = sum(1 for x, y in zip(a, b) if x == y) / n
    pe = sum((sum(1 for x in a if x == v)/n) * (sum(1 for y in b if y == v)/n) for v in (0, 1))
    return round((po - pe) / (1 - pe), 3) if pe < 1 else 0.0

In [ ]:
# estamos sacando solamente el último elemento de cada registro despues lo volvemos binario
VERDICTOS = [int(v) for *_, v in CATALOGO]

# aquí estamos definiendo tres maneras diferentes de decidir si una respuesta es correcta
NIVELES_M = {
    "contiene": lambda instr, sal, esp: niveles(sal, esp)[0], #la respuesta esperada aparece dentro de la respuesta del modelo?
    "estricto": lambda instr, sal, esp: contiene_estricto(sal, esp, instr), #la respuesta del modelo es exactamente igual a la esperada?
    "exacto": lambda instr, sal, esp: niveles(sal, esp)[1], #cuando esperamos la respuesta exacta
}

Probamos los 3 métodos (contiene, estricto, exacto) contra todos los casos del catálogo y medimos qué tan bien coinciden con el veredicto correcto

In [ ]:
print(f"catálogo: {len(CATALOGO)} casos , "
      f"{sum(VERDICTOS)} resueltos y {len(VERDICTOS)-sum(VERDICTOS)} no resueltos\n")
print(f"{'nivel':<10}{'acuerdo':>9}{'kappa':>8}{'falsos+':>9}{'falsos-':>9}")
JUICIOS, KAPPA = {}, {}
for nombre, f in NIVELES_M.items():
    j = [f(i, s, e) for _, i, s, e, _ in CATALOGO]
    JUICIOS[nombre] = j
    KAPPA[nombre] = kappa_cohen(j, VERDICTOS)
    fp = sum(1 for x, y in zip(j, VERDICTOS) if x == 1 and y == 0)
    fn = sum(1 for x, y in zip(j, VERDICTOS) if x == 0 and y == 1)
    ac = sum(1 for x, y in zip(j, VERDICTOS) if x == y) / len(j)
    print(f"{nombre:<10}{ac:>8.0%}{KAPPA[nombre]:>8}{fp:>9}{fn:>9}")

catálogo: 16 casos , 10 resueltos y 6 no resueltos

nivel       acuerdo   kappa  falsos+  falsos-
contiene       81%   0.556        3        0
estricto       94%   0.862        1        0
exacto         75%   0.529        0        4


La tabla compara tres formas de evaluar las respuestas del modelo:

* contiene: tiene buen acuerdo, pero genera falsos positivos porque considera correctas respuestas que solo repiten parte del enunciado.
* exacto: evita falsos positivos, pero genera falsos negativos porque considera * incorrectas respuestas correctas que incluyen texto adicional.
estricto: obtiene el mejor Kappa (0.862) y solo comete un falso positivo, por lo que es el evaluador más equilibrado.


La idea principal es que cada métrica mide algo diferente. exacto mide principalmente obediencia a la instrucción, mientras que estricto intenta evaluar de forma más precisa la respuesta.

Por eso no basta con mirar el accuracy o Kappa: también hay que analizar qué tipo de errores comete cada evaluador. Esto es especialmente importante en LLMOps porque el evaluador determina las conclusiones que sacamos sobre el modelo.

Donde se equivoca cada nivel

In [ ]:
for nombre, j in JUICIOS.items():
    for (etq, instr, sal, esp, v), pred in zip(CATALOGO, j):
        if pred != int(v):
            clase = "falso positivo" if pred else "falso negativo"
            print(f"  {nombre:<9} {clase:<15} [{etq}] salida {sal[:34]!r} esperada {esp!r}")

  contiene  falso positivo  [eco de fecha] salida '2015-03-05' esperada '2015'
  contiene  falso positivo  [eco de frase] salida 'el tren llegó tarde' esperada 'el'
  contiene  falso positivo  [afirma otra] salida 'Tiene 7 letras, no 6 como muchos c' esperada '6'
  estricto  falso positivo  [afirma otra] salida 'Tiene 7 letras, no 6 como muchos c' esperada '6'
  exacto    falso negativo  [verboso] salida 'El resultado es 51.' esperada '51'
  exacto    falso negativo  [con cuentas] salida '36 + 15 = 51' esperada '51'
  exacto    falso negativo  [verboso] salida 'La primera palabra es el.' esperada 'el'
  exacto    falso negativo  [verboso] salida 'La frase tiene 4 palabras.' esperada '4'


Decidiremos con el nivel de mayor kappa, y ese nivel queda
registrado junto con su kappa

In [ ]:
NIVEL_DECISOR = max(KAPPA, key=KAPPA.get)
assert NIVEL_DECISOR == "estricto", f"cambió el mejor nivel: {NIVEL_DECISOR}, revisar el catálogo"
print(f"\nnivel con mayor acuerdo: '{NIVEL_DECISOR}' (kappa {KAPPA[NIVEL_DECISOR]}), es el que decide, y el que vamos a registrar")


nivel con mayor acuerdo: 'estricto' (kappa 0.862), es el que decide, y el que vamos a registrar


### **C) Datos**

* Ocho tipos de instrucción cuya respuesta es exacta y no hay que etiquetar nada a mano, todas son tareas que un modelo de 0.5B ya puede
resolver

* Se reservan cuatro tipos que nunca aparecen en entrenamiento, para ver si la obediencia se transfiere a instrucciones nuevas o solo se memorizó por tipo.

In [ ]:
FRASES = ["el vuelo sale temprano", "hoy hace frío en la ciudad", "el informe está listo",
          "la reunión se pasó al martes", "no funciona el ascensor", "llegaron los repuestos",
          "el servidor está caído", "mañana cierra la inscripción", "el pedido llegó incompleto",
          "se cortó la luz en el laboratorio", "falta firmar el contrato", "el tren viene demorado",
          "la sala está ocupada hasta las seis", "subieron los precios otra vez",
          "el cliente confirmó la fecha", "quedan dos lugares disponibles"]
PALABRAS = ["montaña", "cuaderno", "invierno", "ballena", "pantalla", "recuerdo", "molino",
            "brújula", "cosecha", "linterna", "murciélago", "tormenta", "esquina", "pimienta"]
NOMBRES  = ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo", "Gabriela", "Hugo"]
CIUDADES = [("La Paz", "Bolivia"), ("Lima", "Perú"), ("Quito", "Ecuador"),
            ("Bogotá", "Colombia"), ("Santiago", "Chile"), ("Montevideo", "Uruguay")]

Cada tipo devuelve (instrucción, respuesta correcta), varias redacciones por tipo: el modelo tiene que aprender a obedecer, no a reconocer una frase fija

Pon este texto en mayúsculas: hola
Convierte a mayúsculas lo siguiente: hola
Escribe todo en mayúsculas: hola

In [ ]:
def t_mayusculas(r):
    f = r.choice(FRASES)
    return r.choice([f"Pon este texto en mayúsculas: {f}",
                     f"Convierte a mayúsculas lo siguiente: {f}",
                     f"Escribe todo en mayúsculas: {f}"]), f.upper()
def t_contar_palabras(r):
    f = r.choice(FRASES)
    return r.choice([f"¿Cuántas palabras tiene esta frase? Responde solo el número: {f}",
                     f"Cuenta las palabras y responde solo el número: {f}",
                     f"Dime cuántas palabras hay, solo el número: {f}"]), str(len(f.split()))
def t_primera(r):
    f = r.choice(FRASES)
    return r.choice([f"Responde solo la primera palabra de: {f}",
                     f"Dame únicamente la primera palabra de esta frase: {f}"]), f.split()[0]
def t_comas(r):
    ps = r.sample(PALABRAS, 3)
    return r.choice(["Separa estas palabras con comas: " + " ".join(ps),
                     "Une con comas las siguientes palabras: " + " ".join(ps)]), ", ".join(ps)
def t_si_no(r):
    p, l = r.choice(PALABRAS), r.choice("aeiou")
    return r.choice([f"¿La palabra {p} contiene la letra {l}? Responde solo SI o NO.",
                     f"Responde solo SI o NO: ¿aparece la letra {l} en {p}?"]), "SI" if l in p else "NO"
def t_anio(r):
    a, m, d = r.randint(2015, 2026), r.randint(1, 12), r.randint(1, 28)
    return r.choice([f"Extrae el año de esta fecha y responde solo el número: {a}-{m:02d}-{d:02d}",
                     f"¿De qué año es {a}-{m:02d}-{d:02d}? Solo el número."]), str(a)
def t_suma(r):
    a, b = r.randint(2, 90), r.randint(2, 90)
    return r.choice([f"Suma {a} y {b}. Responde solo el resultado.",
                     f"¿Cuánto es {a} más {b}? Solo el número."]), str(a + b)
def t_json_persona(r):
    n, e = r.choice(NOMBRES), r.randint(18, 70)
    return (f"Devuelve solo un JSON con las claves nombre y edad para: {n}, {e} años.",
            '{"nombre":"%s","edad":%d}' % (n, e))

TIPOS = [t_mayusculas, t_contar_palabras, t_primera, t_comas,
         t_si_no, t_anio, t_suma, t_json_persona]

Tipos que no se entrenan, solo se usan para evaluar transferencia

In [ ]:
def n_ultima(r):
    f = r.choice(FRASES)
    return f"Responde solo la última palabra de: {f}", f.split()[-1]

def n_minusculas(r):
    f = r.choice(FRASES)
    return f"Pon este texto en minúsculas: {f.upper()}", f

def n_contar_letras(r):
    p = r.choice(PALABRAS)
    return f"¿Cuántas letras tiene la palabra {p}? Responde solo el número.", str(len(p))

def n_json_ciudad(r):
    c, pa = r.choice(CIUDADES)
    return (f"Devuelve solo un JSON con las claves ciudad y pais para: {c}, {pa}.",
            '{"ciudad":"%s","pais":"%s"}' % (c, pa))

TIPOS_NUEVOS = [n_ultima, n_minusculas, n_contar_letras, n_json_ciudad]

### **d) filtro**


* Los generadores de arriba producen el par y también producen la respuesta.
* si un generador tiene un error, el error entra al conjunto de entrenamiento como si fuese bueno, el modelo lo aprende y no hay nada en la perdida que avise esto


**Queremos filtrarlo**

Prepararemos dos papeles:

* **generador**: propone un par, a partir de valores y plantillas
* **verificador**: recalcula la respuesta leyendo el enunciado, por un camino distinto, y compara

Un par se acepta solo si el verificador llega a la misma respuesta.


Para que el filtro tenga algo que descartar, se inyectan errores a propósito en algunos
generadores, con una probabilidad fija, sin esto la tasa de descarte saldría 0 % y no se vería nada, con un modelo maestro real, esa tasa aparece sola y suele estar entre el 20 % y el 40 %.

In [ ]:
from collections import Counter

el verificador, cada entrada recalcula la respuesta desde el enunciado

In [ ]:
def _frase(instr):
    return instr.split(": ", 1)[1]

def _v_si_no(instr):
    letra = re.search(r"letra (\w)\b", instr).group(1)
    m = re.search(r"palabra (\w+) contiene", instr) or re.search(r"en (\w+)\?", instr)
    return "SI" if letra in m.group(1) else "NO"

def _v_json_persona(instr):
    m = re.search(r"para: (\w+), (\d+) años", instr)
    return '{"nombre":"%s","edad":%d}' % (m.group(1), int(m.group(2)))

def _v_json_ciudad(instr):
    c, pa = instr.split("para: ", 1)[1].rstrip(".").rsplit(", ", 1)
    return '{"ciudad":"%s","pais":"%s"}' % (c, pa)

VERIFICADORES = {
    "t_mayusculas": lambda i: _frase(i).upper(),
    "t_contar_palabras": lambda i: str(len(_frase(i).split())),
    "t_primera": lambda i: _frase(i).split()[0],
    "t_comas": lambda i: ", ".join(_frase(i).split()),
    "t_si_no":  _v_si_no,
    "t_anio":  lambda i: re.search(r"(\d{4})-\d{2}-\d{2}", i).group(1),
    "t_suma":  lambda i: str(sum(int(x) for x in re.findall(r"\d+", i)[:2])),
    "t_json_persona":  _v_json_persona,
    "n_ultima":  lambda i: _frase(i).split()[-1],
    "n_minusculas":  lambda i: _frase(i).lower(),
    "n_contar_letras":  lambda i: str(len(re.search(r"palabra (\w+)", i).group(1))),
    "n_json_ciudad":  _v_json_ciudad,
}

si se agrega un tipo y se olvida su verificador, esto falla ahora

In [ ]:
assert set(VERIFICADORES) == {t.__name__ for t in TIPOS + TIPOS_NUEVOS}

def verificar(tipo, instr, resp):
    try:
        esperada = VERIFICADORES[tipo](instr)
    except Exception as e:
        return False, f"el verificador no pudo leer el enunciado ({type(e).__name__})"
    if esperada != resp:
        return False, f"dice {resp!r}, se recalcula {esperada!r}"
    return True, "ok"

errores inyectados a propósito, para ver lo que hace el filtro

In [ ]:
P_ERROR = 0.25
ERRORES = {
    "t_anio": lambda i, r: re.findall(r"\d+", i)[-1], # devuelve el día
    "t_contar_palabras": lambda i, r: str(int(r) + 1), # cuenta uno de más
    "t_mayusculas": lambda i, r: r.capitalize(), # solo la inicial
    "t_suma":  lambda i, r: str(int(r) + 10), # falla el acarreo
    "t_json_persona": lambda i, r: r.replace('"edad":', '"anios":'), # renombra la clave
}

Generación con filtro: lo que se descarta es el par, no la instrucción, una instrucción descartada puede volver a salir y esa vez pasar el filtro. Descartar la instrucción agotaría el repertorio de los tipos con pocas frases

In [ ]:
rng, err = random.Random(SEMILLA), random.Random(SEMILLA + 1)
pares, vistos, descartes = [], set(), []
for t in TIPOS:
    n, intentos = 0, 0
    while n < 30 and intentos < 4000:
        intentos += 1
        instr, resp = t(rng)
        if instr in vistos:
            continue
        if t.__name__ in ERRORES and err.random() < P_ERROR:
            resp = ERRORES[t.__name__](instr, resp)
        ok, motivo = verificar(t.__name__, instr, resp)
        if not ok:
            descartes.append({"tipo": t.__name__, "instr": instr, "resp": resp, "motivo": motivo})
            continue
        vistos.add(instr); n += 1
        pares.append({"instr": instr, "resp": resp, "tipo": t.__name__})
    assert n == 30, f"el tipo {t.__name__} solo consiguió {n} pares verificados"

In [ ]:
generados = len(pares) + len(descartes)
TASA_DESCARTE = len(descartes) / generados
print(f"generados {generados} , aceptados {len(pares)} , "
      f"descartados {len(descartes)} ({TASA_DESCARTE:.0%})")
print("descartes por tipo:", dict(Counter(d["tipo"] for d in descartes)))

generados 289 , aceptados 240 , descartados 49 (17%)
descartes por tipo: {'t_mayusculas': 8, 't_contar_palabras': 14, 't_anio': 10, 't_suma': 12, 't_json_persona': 5}


In [ ]:
print("\nun descarte de cada tipo:")
for tipo in dict.fromkeys(d["tipo"] for d in descartes):
    d = next(d for d in descartes if d["tipo"] == tipo)
    print(f"  [{tipo}] {d['instr'][:56]!r}\n      {d['motivo']}")


un descarte de cada tipo:
  [t_mayusculas] 'Pon este texto en mayúsculas: la reunión se pasó al mart'
      dice 'La reunión se pasó al martes', se recalcula 'LA REUNIÓN SE PASÓ AL MARTES'
  [t_contar_palabras] 'Dime cuántas palabras hay, solo el número: llegaron los '
      dice '4', se recalcula '3'
  [t_anio] '¿De qué año es 2024-09-25? Solo el número.'
      dice '25', se recalcula '2024'
  [t_suma] 'Suma 73 y 55. Responde solo el resultado.'
      dice '138', se recalcula '128'
  [t_json_persona] 'Devuelve solo un JSON con las claves nombre y edad para:'
      dice '{"nombre":"Hugo","anios":66}', se recalcula '{"nombre":"Hugo","edad":66}'


El criterio del filtro es parte de la identidad del conjunto, el mismo generador con otro filtro produce otros datos y por lo tanto otro modelo

In [ ]:
CRITERIO_FILTRO = ("verificador independiente por tipo, se descarta el par si la respuesta no coincide con la recalculada desde el enunciado")

un buen filtro no solo descarta datos incorrectos, también permite detectar problemas sistemáticos y auditar la calidad del dataset

---
Convertimos los pares en ejemplos para el modelo

In [ ]:
ejemplos = [dict(armar(p["instr"], p["resp"]), **p) for p in pares]
random.Random(SEMILLA).shuffle(ejemplos)
corte = int(0.9*len(ejemplos))
TRAIN, DEV = ejemplos[:corte], ejemplos[corte:]

DS_ID identifica de forma única el dataset, considerando sus datos, semilla, plantilla y filtro, para garantizar que las comparaciones entre experimentos sean válidas.

In [ ]:
DS_ID = hashlib.sha256(("||".join(sorted(p["instr"]+"|"+p["resp"] for p in pares)) +
                        f"|seed={SEMILLA}|plantilla={PLANTILLA}"
                        f"|filtro={CRITERIO_FILTRO}").encode()).hexdigest()[:12]
msk = sum(sum(1 for l in e["labels"] if l == -100) for e in ejemplos)

print(f"{len(pares)} ejemplos de {len(TIPOS)} tipos , entrenamiento {len(TRAIN)} , "
      f"validación {len(DEV)} | DS_ID {DS_ID}")
print(f"enmascarado {msk/sum(len(e['labels']) for e in ejemplos):.0%} | "
      f"largo máximo {max(len(e['input_ids']) for e in ejemplos)} tokens")
print("\nejemplo de entrenamiento:\n" + PLANTILLA.format(instr=pares[0]["instr"]) + pares[0]["resp"])

240 ejemplos de 8 tipos , entrenamiento 216 , validación 24 | DS_ID 4fecc9806e2d
enmascarado 82% | largo máximo 44 tokens

ejemplo de entrenamiento:
### Instrucción:
Pon este texto en mayúsculas: el pedido llegó incompleto

### Respuesta:
EL PEDIDO LLEGÓ INCOMPLETO


* Esto crea el conjunto de evaluación, asegurándose de que las instrucciones sean nuevas y que las respuestas de referencia sean correctas

* crea una evaluación de 48 ejemplos confiables y sin datos repetidos del entrenamiento, dividida entre tareas conocidas y tareas nuevas

In [ ]:
ev = random.Random(123)
def muestrear(tipos, n, prohibidas):
    fuera, usadas = [], set()
    for _ in range(5000):
        if len(fuera) == n:
            break
        t = ev.choice(tipos)
        i, r_ = t(ev)
        if i not in prohibidas and i not in usadas:
            # El conjunto de evaluación no se filtra: se verifica entero, aqui no hay
            # errores inyectados, así que una falla es un error del código y tiene que
            # detener la corrida
            ok, motivo = verificar(t.__name__, i, r_)
            assert ok, f"referencia inválida en evaluación: {motivo}"
            usadas.add(i); fuera.append((i, r_))
    assert len(fuera) == n, f"solo se consiguieron {len(fuera)} de {n}"
    return fuera

EV_VISTOS = muestrear(TIPOS, 24, vistos) # tipos vistos, instrucciones nuevas
EV_NUEVOS = muestrear(TIPOS_NUEVOS, 24, set())  # tipos que nunca se entrenaron

assert not ({i for i, _ in EV_VISTOS} & vistos), "contaminación con el entrenamiento"
for i, r_ in EV_VISTOS + EV_NUEVOS:
    assert niveles(r_, r_) == (1, 1), i # la referencia se puntúa a sí misma
print(f"evaluación: {len(EV_VISTOS)} de tipos vistos , {len(EV_NUEVOS)} de tipos nuevos , "
      "sin contaminación")

evaluación: 24 de tipos vistos , 24 de tipos nuevos , sin contaminación


El mismo modelo base con 3 ejemplos en el prompt

In [ ]:
PREFIJO_FEWSHOT = "".join(
    PLANTILLA.format(instr=i) + r_ + "\n\n" for i, r_ in
    [("Pon este texto en mayúsculas: buenos días", "BUENOS DÍAS"),
     ("Suma 3 y 4. Responde solo el resultado.", "7"),
     ("Responde solo la primera palabra de: el tren llegó tarde", "el")])
print("el prefijo few-shot cuesta", len(tok(PREFIJO_FEWSHOT)["input_ids"]), "tokens por pedido")

el prefijo few-shot cuesta 79 tokens por pedido


Preparamos los datos para fine-tuning

In [ ]:
ejemplos = [dict(armar(p["instr"], p["resp"]), **p) for p in pares]
random.Random(SEMILLA).shuffle(ejemplos)
corte = int(0.9*len(ejemplos))
TRAIN, DEV = ejemplos[:corte], ejemplos[corte:]

Esto identifica y resume el dataset de entrenamiento: crea un DS_ID único a partir de los datos, semilla, plantilla y filtro, y muestra estadísticas como número de ejemplos, división TRAIN/DEV, porcentaje de tokens enmascarados y longitud máxima

In [ ]:
DS_ID = hashlib.sha256(("||".join(sorted(p["instr"]+"|"+p["resp"] for p in pares)) +
                        f"|seed={SEMILLA}|plantilla={PLANTILLA}"
                        f"|filtro={CRITERIO_FILTRO}").encode()).hexdigest()[:12]
msk = sum(sum(1 for l in e["labels"] if l == -100) for e in ejemplos)

print(f"{len(pares)} ejemplos de {len(TIPOS)} tipos , entrenamiento {len(TRAIN)} , "
      f"validación {len(DEV)} | DS_ID {DS_ID}")
print(f"enmascarado {msk/sum(len(e['labels']) for e in ejemplos):.0%} | "
      f"largo máximo {max(len(e['input_ids']) for e in ejemplos)} tokens")
print("\nejemplo de entrenamiento:\n" + PLANTILLA.format(instr=pares[0]["instr"]) + pares[0]["resp"])

240 ejemplos de 8 tipos , entrenamiento 216 , validación 24 | DS_ID 4fecc9806e2d
enmascarado 82% | largo máximo 44 tokens

ejemplo de entrenamiento:
### Instrucción:
Pon este texto en mayúsculas: el pedido llegó incompleto

### Respuesta:
EL PEDIDO LLEGÓ INCOMPLETO


### e) LoRA

LoRA sobre las proyecciones de atención: con 0.5B y una tarea de formato no hace falta más. Lo que
importa aquí no son los hiperparámetros sino las tres decisiones de arriba —plantilla, máscara de
pérdida y token de fin—, que son las que deciden si esto funciona o produce un modelo que habla
para siempre.

In [ ]:
LORA = LoraConfig(r=16, # tamaño de las matrices pequeñas que LoRA aprende.
                  lora_alpha=32,
                  lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj"] # LoRA se aplica sobre las proyecciones de atención: Query, Key, Value, Output
                  )

HIPER = dict(learning_rate=2e-4,
             num_train_epochs=3,
             per_device_train_batch_size=8,
             gradient_accumulation_steps=2,
             warmup_ratio=0.03, #durante el 3% inicial del entrenamiento aumenta progresivamente el learning rate
             lr_scheduler_type="cosine", # el learning rate disminuye siguiendo una curva coseno
             max_grad_norm=0.3, #limita gradientes demasiado grandes
             optim="adamw_torch")

* El relleno de labels va con -100, no con el token de relleno, y la máscara marca el relleno con 0

* Acá se agrega tokens de padding hasta que todos tengan la misma longitud.

In [ ]:
def colador(lote):
    largo, pad = max(len(e["input_ids"]) for e in lote), tok.pad_token_id
    return {"input_ids": torch.tensor([e["input_ids"]+[pad]*(largo-len(e["input_ids"])) for e in lote]),
            "labels": torch.tensor([e["labels"]+[-100]*(largo-len(e["labels"]))      for e in lote]),
            "attention_mask": torch.tensor([[1]*len(e["input_ids"])+[0]*(largo-len(e["input_ids"])) for e in lote])}

Esta funciónprepara la instrucción, genera una respuesta de forma determinista y devuelve únicamente la respuesta generada, sin repetir el prompt

In [ ]:
def generar(modelo, instr, prefijo=""):
    e = tok(prefijo + PLANTILLA.format(instr=instr), return_tensors="pt",
            add_special_tokens=False).to(modelo.device)
    with torch.no_grad():
        s = modelo.generate(**e, max_new_tokens=48, do_sample=False, temperature=None,
                            top_p=None, top_k=None, pad_token_id=tok.pad_token_id)
    return tok.decode(s[0][e["input_ids"].shape[-1]:], skip_special_tokens=True)

crea un diccionario para guardar las respuestas generadas por cada modelo evaluado

In [ ]:
CRUDAS = {}

Esta función genera respuestas de forma reproducible, evalúa el modelo en tareas vistas y nuevas usando tres métricas, guarda las respuestas y devuelve los resultados para comparar modelos

In [ ]:
def evaluar(modelo, etiqueta, prefijo=""):
    modelo.eval()
    v, salidas = {}, {}

    #evalúa por separado el rendimiento en tareas conocidas y en tareas que el modelo nunca vio durante el entrenamiento
    for nombre, items in [("vistos", EV_VISTOS), ("nuevos", EV_NUEVOS)]:
        sal = [generar(modelo, i, prefijo) for i, _ in items]

        # compara cada respuesta generada con la respuesta esperada para determinar si es correcta
        n = [niveles(s, r_) for s, (_, r_) in zip(sal, items)]
        v[f"{nombre}_contiene"] = [x[0] for x in n]
        v[f"{nombre}_estricto"] = [contiene_estricto(s, r_, i)
                                   for s, (i, r_) in zip(sal, items)]
        v[f"{nombre}_exacto"]   = [x[1] for x in n]
        salidas[nombre] = sal
    CRUDAS[etiqueta] = salidas
    print(f"{etiqueta:<12}" + " ".join(f"{k}={sum(x)}/{len(x)}" for k, x in v.items()))
    return v

verifica que el colador funciona correctamente

In [ ]:
ent = [{"input_ids":e["input_ids"], "labels":e["labels"]} for e in TRAIN]
lote = colador(ent[:4])
assert (lote["labels"][lote["attention_mask"] == 0] == -100).all(), "relleno mal etiquetado"
print("colador verificado ,", tuple(lote["input_ids"].shape))

colador verificado , (4, 39)


cargamos el modelo y lo prepara para hacer fine-tuning con LoRA

In [ ]:
modelo = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16,
                                              device_map={"": 0} if torch.cuda.is_available() else None)
modelo.gradient_checkpointing_enable()
modelo.enable_input_require_grads()   # sin esto, el backward falla
modelo = get_peft_model(modelo, LORA)
for _, p in modelo.named_parameters():
    if p.requires_grad:
        p.data = p.data.float() # adaptadores en fp32

El base se mide antes de entrenar, con los adaptadores apagados: es el mismo objeto, así que la comparación es contra exactamente esos pesos.

mide el modelo original en dos condiciones:

* base: sin ejemplos adicionales
* base+3ej: usando 3 ejemplos few-shot en el prompt

In [ ]:
VEC = {}

with modelo.disable_adapter():
    VEC["base"] = evaluar(modelo, "base")
    VEC["base+3ej"] = evaluar(modelo, "base+3ej", PREFIJO_FEWSHOT)

base        vistos_contiene=16/24 vistos_estricto=4/24 vistos_exacto=0/24 nuevos_contiene=11/24 nuevos_estricto=1/24 nuevos_exacto=1/24
base+3ej    vistos_contiene=18/24 vistos_estricto=11/24 vistos_exacto=17/24 nuevos_contiene=10/24 nuevos_estricto=1/24 nuevos_exacto=6/24


Esto define cómo se realizará el entrenamiento: precisión fp16 si hay GPU, semillas reproducibles, logging, gradient checkpointing y los hiperparámetros definidos anteriormente

In [ ]:
args = TrainingArguments(output_dir="salida",
                         fp16=torch.cuda.is_available(),
                         seed=SEMILLA,
                         data_seed=SEMILLA,
                         logging_steps=5,
                         report_to=[],
                         gradient_checkpointing=True,
                         remove_unused_columns=False,
                         save_strategy="no",
                         **HIPER)

trainer = Trainer(model=modelo, args=args, train_dataset=ent, data_collator=colador)

#medimos tiempo
t0 = time.time()
hist = trainer.train()
segundos = time.time() - t0

#guardamos
modelo.save_pretrained("adaptador")

# Creamos métricas del entrenamiento
MED = {"perdida_final": round(hist.training_loss, 4), "segundos": round(segundos, 1),
       "seg_por_paso": round(segundos/max(hist.global_step, 1), 2),
       "memoria_pico_gb": round(torch.cuda.max_memory_allocated()/1e9, 2) if torch.cuda.is_available() else 0,
       "entrenables": sum(p.numel() for p in modelo.parameters() if p.requires_grad)}
print(MED)
VEC["ajustado"] = evaluar(modelo, "ajustado")

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
5,0.783700
10,0.281700
15,0.164800
20,0.115900
25,0.106800
30,0.116100
35,0.081600


{'perdida_final': 0.2189, 'segundos': 19.4, 'seg_por_paso': 0.5, 'memoria_pico_gb': 2.3, 'entrenables': 2162688}
ajustado    vistos_contiene=21/24 vistos_estricto=14/24 vistos_exacto=21/24 nuevos_contiene=8/24 nuevos_estricto=4/24 nuevos_exacto=8/24


establece los tres modelos/estrategias que se comparan:

In [ ]:
SIS  = ["base","base+3ej","ajustado"]

verificamos que VEC contiene resultados para los tres sistemas antes de continuar

In [ ]:
assert set(SIS) <= set(VEC), f"faltan en VEC: {set(SIS)-set(VEC)}"

Lo que contestó literalmente cada sistema

In [ ]:
for k in (0, 1, 2):
    print(f"\n>>> {EV_VISTOS[k][0]}")
    print(f"    esperado : {EV_VISTOS[k][1]}")
    for s in SIS:
        print(f"    {s:<9}: {CRUDAS[s]['vistos'][k][:100]!r}")


>>> Suma 36 y 15. Responde solo el resultado.
    esperado : 51
    base     : '12'
    base+3ej : '51\n\n### Instrucción:\nResponde solo la primera palabra de: el tren llegó tarde\n\n### Respuesta:\nel\n\n##'
    ajustado : '51'

>>> ¿Cuánto es 70 más 73? Solo el número.
    esperado : 143
    base     : '70 + 73 = 143'
    base+3ej : '143\n\n### Instrucción:\n¿Cuánto es 70 menos 73? Solo el número.\n\n### Respuesta:\n-3\n\n### Instrucción:\n¿'
    ajustado : '143'

>>> ¿De qué año es 2015-03-05? Solo el número.
    esperado : 2015
    base     : '2015-03-05\n\n### Ejercicio 2:\n¿Cuál es el día de la semana en el que se cumple el 15 de marzo de 2015'
    base+3ej : '2015\n\n### Instrucción:\n¿Cuál es el número de la semana en que el 15 de marzo de 2015 es el día de la'
    ajustado : '2015'


### f) El registro en MLflow

Convertimos el fine-tuning en un experimento reproducible y trazable: registra parámetros, pérdida, métricas y adaptador en MLflow, vincula el modelo con su DS_ID, verifica esa relación y empaqueta todos los artefactos

In [ ]:
from mlflow.tracking import MlflowClient
cliente, MODELO = MlflowClient(), "instruct-qwen0.5b"

with mlflow.start_run(run_name="instruction-tuning") as run:
    mlflow.log_params({"base": BASE, "ds_id": DS_ID, "semilla": SEMILLA, "r": LORA.r,
                       "lora_alpha": LORA.lora_alpha, "n_ejemplos": len(TRAIN),
                       "tipos_entrenados": len(TIPOS),
                       "target_modules": ",".join(sorted(LORA.target_modules)),
                       # La plantilla es parte del artefacto
                       "plantilla": PLANTILLA,
                       # El criterio del filtro y el nivel decisor son parte de la
                       # receta, sin ellos el conjunto y la tabla no se reproducen.
                       "criterio_filtro": CRITERIO_FILTRO,
                       "nivel_decisor": NIVEL_DECISOR, **HIPER})
    # La curva, paso a paso, como la registró el Trainer.
    for h in trainer.state.log_history:
        if "loss" in h:
            mlflow.log_metric("perdida", h["loss"], step=int(h["step"]))
    # mlflow no acepta "+" en el nombre de una métrica: "base+3ej" la rechaza entera.
    limpio = lambda s: s.replace("+", "_")
    mlflow.log_metrics({**MED,
                        # La calidad de los datos y la del instrumento se registran
                        # como métricas del run, igual que el acierto.
                        "tasa_descarte": TASA_DESCARTE,
                        "n_descartados": len(descartes),
                        **{f"kappa_{k}": v for k, v in KAPPA.items()},
                        **{f"{eje}_{limpio(s)}": sum(VEC[s][eje])/len(VEC[s][eje])
                           for eje in EJES for s in SIS}})
    mlflow.log_artifacts("adaptador", artifact_path="adaptador")
    run_id = run.info.run_id

ver = mlflow.register_model(f"runs:/{run_id}/adaptador", MODELO)
cliente.set_model_version_tag(MODELO, ver.version, "ds_id", DS_ID)
# La plantilla viaja con la versión: es lo primero que se pierde al desplegar, y sin
# ella el adaptador produce texto plausible y equivocado.
cliente.set_model_version_tag(MODELO, ver.version, "plantilla", PLANTILLA)
cliente.set_registered_model_alias(MODELO, "campeon", ver.version)

# La verificación que justifica todo el registro.
v = cliente.get_model_version_by_alias(MODELO, "campeon")
assert v.tags["ds_id"] == DS_ID, "el adaptador promovido no corresponde a este conjunto"
print(f"{MODELO} v{v.version} | alias 'campeon' | ds_id {v.tags['ds_id']} | verificado")

# Para ver la interfaz de MLflow dentro de Colab, sin túnel externo:
get_ipython().system_raw("mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000 &")

import time
time.sleep(5)

from google.colab import output
output.serve_kernel_port_as_window(5000)

json.dump({"VEC": VEC, "CRUDAS": CRUDAS, "MED": MED, "DS_ID": DS_ID},
          open("resultados.json","w"), ensure_ascii=False)
!zip -qr artefactos.zip resultados.json adaptador mlflow.db mlruns
try:
    from google.colab import files; files.download("artefactos.zip")
except ImportError:
    print("fuera de Colab: descarga artefactos.zip a mano.")

Registered model 'instruct-qwen0.5b' already exists. Creating a new version of this model...
Created version '2' of model 'instruct-qwen0.5b'.


instruct-qwen0.5b v2 | alias 'campeon' | ds_id 4fecc9806e2d | verificado
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>